In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-02-27 17:08:07 | people | execute | Started
2025-02-27 17:08:07 | people | load | Started
2025-02-27 17:08:13 | people | load | Completed in 0.1 min
2025-02-27 17:08:13 | people | transform | Started
2025-02-27 17:08:13 | people | transform | Completed in 0.0 min
2025-02-27 17:08:13 | people | write | Started
2025-02-27 17:09:07 | people | write | Completed in 0.88 min
2025-02-27 17:09:07 | people | execute | Completed in 1.0 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-02-27 17:08:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|               C-3PO|  2|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|               R2-D2|  3|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|         Darth Vader|  4|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|         Leia Organa|  5|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|           Owen Lars|  6|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|  Beru Whitesun lars|  7|https://www.swapi...|{"created": "2025...|
|2025-02-27 17:08:...|               R5-D4|  8|https://www.swapi...|{"created": "2025..

# 2 Silver

In [11]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [13]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            df = self.transf_people(df)
        return df

    def transf_people(self, df: DataFrame) -> DataFrame:
        df = (
            df.withColumn("height", df.properties.height)
            .withColumn("mass", df.properties.mass)
            .withColumn("gender", df.properties.gender)
            .drop("url", "properties")
        )
        for i in range(0, 35):
            df = df.withColumn(f"col{str(i)}", F.lit(str(i)))
        return df


silver_instance = StarWarsSilver(spark, **options)

In [14]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "people"
)

2025-02-27 17:09:17 | people | execute | Started
2025-02-27 17:09:17 | people | load | Started
2025-02-27 17:09:17 | people | load | Completed in 0.0 min
2025-02-27 17:09:17 | people | transform | Started
2025-02-27 17:09:18 | people | transform | Completed in 0.02 min
2025-02-27 17:09:18 | people | write | Started
2025-02-27 17:09:25 | people | write | Completed in 0.1 min
2025-02-27 17:09:25 | people | execute | Completed in 0.13 min


In [15]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+-------------------------+-------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|LH_SilverTS              |LH_BronzeTS              |name                 |uid|height |mass   |gender       |col0|col1|col2|col3|col4|col5|col6|col7|col8|col9|col10|col11|col12|col13|col14|col15|col16|col17|col18|col19|col20|col21|col22|col23|col24|col25|col26|col27|col28|col29|col30|col31|col32|col33|col34|
+-------------------------+-------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|2025-02-27 17:09:19.35517|2025-02-27 17:08:15.26651|Clie

# 3 Optimize Silver

In [16]:
# Set layout for liquid columns and optimize undependend of write operation
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-02-27 17:09:30 | people | execute | Started
2025-02-27 17:09:30 | people | load | Started
2025-02-27 17:09:30 | people | load | Completed in 0.0 min
2025-02-27 17:09:30 | people | transform | Started
2025-02-27 17:09:31 | people | transform | Completed in 0.0 min
2025-02-27 17:09:31 | people | write | Started
2025-02-27 17:09:36 | people | write | Completed in 0.08 min
2025-02-27 17:09:36 | people | tblproperties | Started
2025-02-27 17:10:06 | people | tblproperties | Completed in 0.48 min
2025-02-27 17:10:06 | people | optimize | Started
2025-02-27 17:10:55 | people | optimize | Completed in 0.82 min
2025-02-27 17:10:55 | people | execute | Completed in 1.4 min


In [17]:
# You can also do this with the default class
(
    silver.Silver(spark, **options)
    .tblproperties(clusterby=["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-02-27 17:10:55 | people | execute | Started
2025-02-27 17:10:55 | people | tblproperties | Started
2025-02-27 17:10:56 | people | tblproperties | Completed in 0.02 min
2025-02-27 17:10:56 | people | optimize | Started
2025-02-27 17:11:31 | people | optimize | Completed in 0.57 min
2025-02-27 17:11:31 | people | execute | Completed in 0.58 min


In [18]:
# run above commands also together with the write
(
    silver_instance.load()
    .transform()
    .write(mode="overwrite", merge_schema=True)
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-02-27 17:11:31 | people | execute | Started
2025-02-27 17:11:31 | people | load | Started
2025-02-27 17:11:31 | people | load | Completed in 0.0 min
2025-02-27 17:11:31 | people | transform | Started
2025-02-27 17:11:31 | people | transform | Completed in 0.0 min
2025-02-27 17:11:31 | people | write | Started
2025-02-27 17:11:36 | people | write | Completed in 0.08 min
2025-02-27 17:11:36 | people | tblproperties | Started
2025-02-27 17:11:40 | people | tblproperties | Completed in 0.05 min
2025-02-27 17:11:40 | people | optimize | Started
2025-02-27 17:12:17 | people | optimize | Completed in 0.62 min
2025-02-27 17:12:17 | people | execute | Completed in 0.77 min


In [19]:
# run also a FULL Optimize when liquid clustering and a vacuum Lite with Delta 3.3 and above
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(optimize=True, optimize_full=True, vacuum=True, vacuum_lite=True)
    .execute("people")
)

2025-02-27 17:12:17 | people | execute | Started
2025-02-27 17:12:17 | people | load | Started
2025-02-27 17:12:17 | people | load | Completed in 0.0 min
2025-02-27 17:12:17 | people | transform | Started
2025-02-27 17:12:17 | people | transform | Completed in 0.0 min
2025-02-27 17:12:17 | people | write | Started
2025-02-27 17:12:23 | people | write | Completed in 0.08 min
2025-02-27 17:12:23 | people | tblproperties | Started
2025-02-27 17:12:27 | people | tblproperties | Completed in 0.05 min
2025-02-27 17:12:27 | people | optimize | Started
2025-02-27 17:12:43 | people | optimize | Completed in 0.27 min
2025-02-27 17:12:43 | people | execute | Completed in 0.43 min


In [ ]:
# Set besides cluster cols, some integrated delta tblproperties and any other delta tblproperties as documented on https://docs.delta.io/
(
    silver_instance.tblproperties(
        clusterby=["gender"], 
        deletion_vectors=True,
        auto_compact=True,
        optimize_write=True,
        change_data_feed=True,
        row_tracking=True,
        type_widening=True,
        tblproperties={"delta.enableChangeDataFeed": "true"}
    ).execute("people")
)

2025-02-27 17:12:43 | people | execute | Started
2025-02-27 17:12:43 | people | load | Started
2025-02-27 17:12:43 | people | load | Completed in 0.0 min
2025-02-27 17:12:43 | people | transform | Started
2025-02-27 17:12:43 | people | transform | Completed in 0.0 min
2025-02-27 17:12:43 | people | write | Started
2025-02-27 17:12:48 | people | write | Completed in 0.07 min
2025-02-27 17:12:48 | people | tblproperties | Started
2025-02-27 17:12:51 | people | tblproperties | Completed in 0.05 min
2025-02-27 17:12:51 | people | optimize | Started
2025-02-27 17:13:09 | people | optimize | Completed in 0.28 min
2025-02-27 17:13:09 | people | execute | Completed in 0.42 min


In [21]:
# Run analyze and exclude cols you dont want an analyze to run. Especially long strings.
# Does not work with spark_catalog in Spark.
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(analyze=True, excl_cols=["url"])
    .execute("people")
)

2025-02-27 17:13:09 | people | execute | Started
2025-02-27 17:13:09 | people | load | Started
2025-02-27 17:13:09 | people | load | Completed in 0.0 min
2025-02-27 17:13:09 | people | transform | Started
2025-02-27 17:13:09 | people | transform | Completed in 0.0 min
2025-02-27 17:13:09 | people | write | Started
2025-02-27 17:13:14 | people | write | Completed in 0.08 min
2025-02-27 17:13:14 | people | tblproperties | Started
2025-02-27 17:13:18 | people | tblproperties | Completed in 0.05 min
2025-02-27 17:13:18 | people | optimize | Started


AssertionError: Column url does not exist in the table spark_catalog.silver.people

# Review in Delta History and details

In [22]:
from delta.tables import *

q = f"DESCRIBE HISTORY {CATALOG}.silver.people"
hist = spark.sql(q)
hist.show(truncate=False)

q = f"DESCRIBE DETAIL {CATALOG}.silver.people"
det = spark.sql(q)
det.show(truncate=False)

q = f"SHOW TBLPROPERTIES {CATALOG}.silver.people"
prop = spark.sql(q)
prop.show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp        

# 4 Clean Up

In [23]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")

DataFrame[]